In [ ]:
# 1. Instal·lació de dependències per a Google Colab
%%shell
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
apt-get update
apt-get install -y google-chrome-stable
pip install selenium==4.18.1 webdriver-manager
google-chrome-stable --version

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote
import glob
import os
import time
import shutil
import pandas as pd
from google.colab import files

In [ ]:
def web_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("--window-size=1920,1200")
    
    # Forcem la carpeta de descàrregues a /content/
    prefs = {"download.default_directory": "/content/"}
    options.add_experimental_option("prefs", prefs)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

In [ ]:
def descarrega_xlsx(nou_nom):
    # Busca tots els arxius xlsx a /content/ que NO siguin el fitxer d'entitats
    xlsx_files = glob.glob('/content/brand_db_export*.xlsx')
    if not xlsx_files:
         # Per si el nom de l'export ha canviat
         xlsx_files = [f for f in glob.glob('/content/*.xlsx') if f not in ['/content/entitats.xlsx', '/content/log.xlsx']]
    
    if not xlsx_files:
        return False

    # Agafa el més recent
    xlsx_files.sort(key=os.path.getmtime, reverse=True)
    ultim_arxiu = xlsx_files[0]

    wipo_folder = '/content/wipo'
    if not os.path.exists(wipo_folder):
        os.makedirs(wipo_folder)
    
    # Substituïm caràcters especials en el nom del fitxer
    safe_name = nou_nom.replace('/', '_').replace('\\', '_').replace(':', '_')
    nou_nom_complet = os.path.join(wipo_folder, safe_name + ".xlsx")
    
    try:
        if os.path.exists(nou_nom_complet): os.remove(nou_nom_complet)
        shutil.move(ultim_arxiu, nou_nom_complet)
        return True
    except Exception as e:
        print(f"      Error al desar el fitxer: {e}")
        return False

In [ ]:
def titular(driver, codi):
  print(f"Processant: {codi}...")
  # Fem cerca exacta utilitzant cometes
  val_query = quote(f'"{codi}"')
  url = f"https://branddb.wipo.int/en/advancedsearch/results?sort=score%20desc&strategy=concept&asStructure=%7B%22_id%22:%22aff0%22,%22boolean%22:%22AND%22,%22bricks%22:%5B%7B%22_id%22:%22aff1%22,%22key%22:%22applicant%22,%22strategy%22:%22Simple%22,%22value%22:%22{val_query}%22%7D%5D%7D"

  driver.get(url)

  try:
    # Comprovem si el botó de descàrrega apareix O si no hi ha resultats
    wait = WebDriverWait(driver, 30)
    download_xpath = '//span[text()="Download results"]'
    
    # Aquesta espera acaba si troba el botó O si sap segur que no n'hi haurà
    wait.until(lambda d: d.find_elements(By.XPATH, download_xpath) or 
                         "Displaying 0-0" in d.page_source or 
                         "No records found" in d.page_source)
    
    if not driver.find_elements(By.XPATH, download_xpath):
        print(f"   ! Cap marca trobada per a {codi}")
        return codi, "No trobat"

    # Si ha arribat aquí, hi ha resultats
    try:
        btn = driver.find_element(By.XPATH, download_xpath)
        btn.click()
    except Exception:
        # Intent alternatiu si el clic de Selenium falla
        driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, download_xpath))
    
    excel_opt = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable((By.XPATH, '//li//span[text()="EXCEL"]'))
    )
    excel_opt.click()

    # Temps d'espera per la descàrrega (incrementat a 12s per seguretat)
    time.sleep(12)

    if descarrega_xlsx(codi):
        print(f"   OK: Descarregada informació de {codi}")
        return codi, "1"
    else:
        print(f"   ? Error al detectar el fitxer de {codi}")
        return codi, "Error fitxer"

  except TimeoutException:
    print(f"   Timeout per a: {codi}")
    return codi, "Timeout"
  except Exception as e:
    print(f"   Error amb {codi}: {type(e).__name__}")
    return codi, f"Error {type(e).__name__}"

In [ ]:
# Carregar el dataset (assegura't de tenir el fitxer entitats.xlsx pujat a Colab)
if os.path.exists("entitats.xlsx"):
    df = pd.read_excel("entitats.xlsx")
    # Neteja de noms per a la cerca (entitat principal)
    # Eliminem tot el que hi hagi després d'una coma o un parèntesi per a la cerca
    entities = df['Titular (entitat)'].apply(lambda x: str(x).rsplit(",", 1)[0].rsplit(' (', 1)[0].strip()).unique()

    log = []
    print(f"Iniciant driver per a {len(entities)} entitats...")
    driver = web_driver()
    
    try:
        for ent in entities:
            entitat, status = titular(driver, ent)
            log.append([entitat, status])
    finally:
        driver.quit()
        pd.DataFrame(log, columns=["Entitat", "Estat"]).to_excel("log_results.xlsx", index=False)
        print("\nProcés finalitzat. Resultats guardats a log_results.xlsx")
else:
    print("Error: Puja el fitxer 'entitats.xlsx' a la carpeta /content/")

In [ ]:
# Empaquetar tot el resultat en un ZIP per baixar-lo de cop
if os.path.exists('/content/wipo'):
    # Eliminem el zip si ja existia per actualitzar-lo
    if os.path.exists('marques_wipo_export.zip'): os.remove('marques_wipo_export.zip')
    shutil.make_archive('marques_wipo_export', 'zip', '/content/wipo')
    files.download('marques_wipo_export.zip')
    print("Descàrrega de ZIP iniciada.")